In [5]:
import os

import json
import csv
import textwrap
from pathlib import Path
from datetime import datetime

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
# from langchain_text_splitters import RecursiveCharacterSplitter
# from langchain.text_splitters import RecursiveCharacterSplitter


from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# from langchain_text_splitters import RecursiveCharacterSplitter

In [7]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

In [8]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
llm = ChatOpenAI(model='gpt-4o-mini')

In [15]:
embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')

In [11]:
SAMPLE_DIR = Path('sample_data')
SAMPLE_DIR.mkdir(exist_ok=True)

In [12]:
sample_texts = {
    'company_policy.txt': """주식회사 모두의연구소 사내 규정

제1조 (목적)
이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.

제2조 (근무시간)
1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.
2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.
3. 재택근무는 주 2회까지 가능하다.

제3조 (휴가)
1. 연차휴가는 근로기준법에 따라 부여한다.
2. 경조사 휴가는 별도 규정에 따른다.
3. 자기개발 휴가를 연 5일 추가 부여한다.

제4조 (교육)
1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.
2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.
3. 온라인 학습 플랫폼 이용료를 전액 지원한다.
""",
    'ai_report.txt': """2024년 인공지능 산업 동향 보고서

1. 개요
2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.
특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.

2. 주요 트렌드
- RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.
- 멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.
- AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.
- 소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.

3. 시장 전망
2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.
특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.
""",
    'product_manual.txt': """스마트 홈 허브 v3.0 사용자 매뉴얼

1. 제품 소개
스마트 홈 허브 v3.0은 AI 기반 홈 자동화 컨트롤러입니다.
음성 인식, 자동 스케줄링, 에너지 최적화 기능을 제공합니다.

2. 초기 설정
Step 1: 전원을 연결하고 Wi-Fi 네트워크에 접속합니다.
Step 2: 모바일 앱을 설치하고 QR 코드를 스캔합니다.
Step 3: 연동할 IoT 기기를 검색하고 등록합니다.

3. 주요 기능
- 음성 명령: "허브야, 거실 조명 켜줘" 등의 자연어 명령 지원
- 자동 스케줄: 시간대별 기기 자동 제어
- 에너지 모니터링: 실시간 전력 사용량 확인 및 절약 팁 제공
- 보안 모드: 외출 시 자동 보안 설정
"""
}

In [13]:
for filename, content in sample_texts.items():
    (SAMPLE_DIR / filename).write_text(content, encoding='utf-8')
    # sample_data/product_manual.txt

In [14]:
csv_data = [
    {'이름' :'김철수', '부서' : '개발팀', '직급' : '선임'},
    {'이름' :'이영희', '부서' : '기획팀', '직급' : '매니저'},
    {'이름' :'박지민', '부서' : '개발팀', '직급' : '주임'},
]

with open(SAMPLE_DIR / 'employees.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['이름', '부서', '직급'])
    writer.writeheader()
    writer.writerows(csv_data)

#### RAG : Retrieval Augmented Generation
#### Fine-tuning : GPT를 새로 교육시킴

In [16]:
knowledge_base = [
    {'id': 1, 'content': '파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.', 'source': 'programming_guide.txt'},
    {'id': 2, 'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.', 'source': 'ai_glossary.txt'},
    {'id': 3, 'content': '벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.', 'source': 'database_manual.txt'},
]

query = 'RAG 기술이 무엇인가요?'

def retrieve_by_similarity(query, documents):
    contents = [doc['content'] for doc in documents]
    
    query_vec = embeddings.embed_query(query)
    doc_vecs = embeddings.embed_documents(contents)
    
    scores = cosine_similarity([query_vec], doc_vecs)[0]
    
    ranked = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return [(doc, float(score)) for doc, score in ranked]

retrieve_by_similarity(query, knowledge_base)

[({'id': 2,
   'content': 'RAG는 Retrieval-Augmented Generation의 약자로, LLM이 외부 데이터를 참조하여 답변을 생성하는 기술입니다.',
   'source': 'ai_glossary.txt'},
  0.558964657265516),
 ({'id': 1,
   'content': '파이썬은 1991년 귀도 반 로섬이 만든 프로그래밍 언어입니다. 간결한 문법과 풍부한 라이브러리가 특징입니다.',
   'source': 'programming_guide.txt'},
  0.25852186395932497),
 ({'id': 3,
   'content': '벡터 데이터베이스는 텍스트를 숫자 벡터로 변환하여 저장하고, 유사도 검색을 빠르게 수행하는 데이터베이스입니다.',
   'source': 'database_manual.txt'},
  0.17958358236956232)]

In [18]:
# class : variables, methods, 하나로 묶는 단위 (설계도)
# class classname:
    
#     def __init__():
        
#     def
class Calculator:
    def __init__(self):
        self.history = []
        
    def add(self, a, b):
        result = a+b
        self.history.append(f'{a} + {b} = {result}')
        return result
    
    def get_history(self):
        return self.history

In [19]:
calc = Calculator()

In [20]:
calc.history

[]

In [22]:
calc.add(1, 2)

3

In [23]:
calc.history

['1 + 2 = 3']

In [24]:
calc.get_history()

['1 + 2 = 3']

In [51]:
class DocumentStore:
    def __init__(self):
        self.documents = []
        self.next_id = 1
        self.embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')
        
    def add(self, content, source='unknown'):
        self.documents.append({
            'id' : self.next_id,
            'content' : content,
            'source' : source
        })
        self.next_id +=1
        
    def count(self):
        return len(self.documents)
    
    def search(self, keyword):
        return [doc for doc in self.documents if keyword in doc['content']]
    
    def retrieve(self, query, top_k=3):
        if not self.documents:
            return []
        
        embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')
        contents = [doc['content'] for doc in self.documents]
    
        query_vec = embeddings.embed_query(query)
        doc_vecs = embeddings.embed_documents(contents)

        scores = cosine_similarity([query_vec], doc_vecs)[0]

        ranked = sorted(zip(self.documents, scores), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]
    
    def generate(self, query):
        retrieved = self.retrieve(query)
        
        if not retrieved:
            return "관련 문서를 찾을 수 없습니다."
        
        contexts = '\n'.join(f"[문서 {d[0]['id']}, 출처 : {d[0]['source']}] {d[0]['content']}" for d in retrieved)
        messages = [
            SystemMessage(content = '제공된 문서를 기반으로 정확하게 답변해주세요. 출처를 명시해주세요'),
            HumanMessage(content = f'참고 문서:\n{contexts}\n\n질문: {query}')
        ]
        response = llm.invoke(messages)
        return response.content

In [52]:
store = DocumentStore()
store.add('파이썬은 데이터 분석에 좋다', 'guide.txt')
store.add('RAG는 검색증강 생성이다', 'glossary.txt')

In [53]:
store.generate('rag에 대해서 설명해주세요')

'RAG는 "검색증강 생성"의 약자로, 검색과 생성의 두 가지 기능을 결합한 기법입니다. 이 방법은 주로 정보 검색과 자연어 처리 분야에서 사용되며, 특정 질문에 대한 답변을 제공하기 위해 검색된 정보를 활용하여 더 나은 결과를 생성하는 방식입니다. \n\n출처: [문서 2, 출처 : glossary.txt]'

In [28]:
store.count()

2

In [30]:
store.search('파이썬')

[{'id': 1, 'content': '파이썬은 데이터 분석에 좋다', 'source': 'guide.txt'}]

In [33]:
from collections import Counter

class WordCounter:
    def __init__(self):
        self.texts = []
    
    def add_text(self, text) :
        self.texts.append(text)
        
    def count_words(self):
        return sum(len(t.split()) for t in self.texts)
    
    def most_common(self, n):
        all_words = []
        for t in self.texts:
            all_words.extend(t.split())
        return Counter(all_words).most_common(n)
    

In [32]:
wc = WordCounter()
wc.add_text('파이썬은 어렵지 않습니다')
wc.add_text('나는 학교에 갑니다')

wc.count_words()

6

In [34]:
all_words = [1,1,1,1,1, 2,2,2,2, 3,3,3, 4,4,5]
Counter(all_words)

Counter({1: 5, 2: 4, 3: 3, 4: 2, 5: 1})

In [36]:
Counter(all_words).most_common(2)

[(1, 5), (2, 4)]

In [44]:
docs = ['rag는 검색증강생성 기술입니다', '외부 문서를 검색해서 llm답변에 활용합니다']
def rag_with_langchain(query, documents):
    if not documents:
        return '참고할 문서가 없습니다'
    
    contexts = '\n'.join(f"- {doc}" for doc in documents)
    messages = [
        SystemMessage(content = '제공된 문서를 기반으로 정확하게 답변해주세요'),
        HumanMessage(content = f'참고 문서:\n{contexts}\n\n질문: {query}')
    ]
    response = llm.invoke(messages)
    return response.content

In [45]:
rag_with_langchain('rag가 뭐야?', docs)

'RAG는 "Retrieval-Augmented Generation"의 약자로, 검색증강생성 기술입니다. 이 기술은 외부 문서를 검색하여 얻은 정보를 활용해 대규모 언어 모델(LLM)의 답변 품질을 향상시키는 방법입니다. 즉, 모델이 생성하는 결과에 필요한 정보를 검색하여 이를 보강하는 방식으로 작동합니다.'

In [46]:
rag_with_langchain('rag가 뭐야?', [])

'참고할 문서가 없습니다'

In [54]:
from langchain_core.documents import Document

In [ ]:
doc = Document(page_content = "문서 내용...", metadata ={"source" : "file.txt", "type":"policy"})

In [55]:
file_path = SAMPLE_DIR / 'company_policy.txt'

with open(file_path, 'r', encoding='utf-8') as f:
    raw_text = f.read()

In [58]:
file_path.name

'company_policy.txt'

In [56]:
raw_text

'주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n1. 연차휴가는 근로기준법에 따라 부여한다.\n2. 경조사 휴가는 별도 규정에 따른다.\n3. 자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n3. 온라인 학습 플랫폼 이용료를 전액 지원한다.\n'

In [59]:
doc = Document(page_content = raw_text, 
               metadata ={"source" : file_path.name, 
                          "file_type" : "txt",
                          "char_count" : len(raw_text),
                          "line_count" :len(raw_text.splitlines())
                         }
              )

In [60]:
doc

Document(metadata={'source': 'company_policy.txt', 'file_type': 'txt', 'char_count': 383, 'line_count': 19}, page_content='주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n1. 연차휴가는 근로기준법에 따라 부여한다.\n2. 경조사 휴가는 별도 규정에 따른다.\n3. 자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n3. 온라인 학습 플랫폼 이용료를 전액 지원한다.\n')

In [63]:
def load_text_files(directory):
    documents = []
    for fp in sorted(directory.glob('*.txt')):
        text = fp.read_text(encoding='utf-8')
        doc = Document(page_content = text, 
               metadata ={"source" : fp.name,
                          "char_count" : len(text),
                         }
              )
        documents.append(doc)
    
    return documents

In [64]:
all_docs = load_text_files(SAMPLE_DIR)

In [73]:
all_docs

[Document(metadata={'source': 'ai_report.txt', 'char_count': 413}, page_content='2024년 인공지능 산업 동향 보고서\n\n1. 개요\n2024년 인공지능 산업은 전년 대비 35% 성장하여 약 5,000억 달러 규모에 도달했다.\n특히 생성형 AI 분야가 전체 성장의 60%를 견인했다.\n\n2. 주요 트렌드\n- RAG(Retrieval-Augmented Generation): 기업용 AI 솔루션의 핵심 기술로 자리잡았다.\n- 멀티모달 AI: 텍스트, 이미지, 음성을 통합 처리하는 모델이 확산되었다.\n- AI 에이전트: 자율적으로 작업을 수행하는 AI 에이전트 시장이 급성장했다.\n- 소형 언어 모델(SLM): 경량화된 모델로 온디바이스 AI가 확대되었다.\n\n3. 시장 전망\n2025년에는 AI 산업이 약 7,000억 달러 규모로 성장할 것으로 예상된다.\n특히 RAG 기반 엔터프라이즈 솔루션 시장이 크게 확대될 전망이다.\n'),
 Document(metadata={'source': 'company_policy.txt', 'char_count': 383}, page_content='주식회사 모두의연구소 사내 규정\n\n제1조 (목적)\n이 규정은 주식회사 모두의연구소의 임직원이 준수해야 할 기본적인 사항을 정하는 것을 목적으로 한다.\n\n제2조 (근무시간)\n1. 기본 근무시간은 오전 9시부터 오후 6시까지로 한다.\n2. 유연근무제를 시행하며, 코어타임은 오전 10시부터 오후 4시까지이다.\n3. 재택근무는 주 2회까지 가능하다.\n\n제3조 (휴가)\n1. 연차휴가는 근로기준법에 따라 부여한다.\n2. 경조사 휴가는 별도 규정에 따른다.\n3. 자기개발 휴가를 연 5일 추가 부여한다.\n\n제4조 (교육)\n1. 모든 임직원은 연간 40시간 이상의 교육을 이수해야 한다.\n2. 외부 컨퍼런스 참석비를 연 200만원까지 지원한다.\n3. 온라인 학습 플랫폼 이용료를 전액 지원한다.\

In [71]:
def load_csv_file(file_path):
    documents = []
    with open(file_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            content = ' | '.join(f'{k}: {v}' for k, v in row.items() )
            doc = Document(page_content = content, 
               metadata ={"source" : file_path.name,
                          "char_count" : len(content),
                         }
              )
            documents.append(doc)
    return documents

In [72]:
csv_docs = load_csv_file(SAMPLE_DIR / 'employees.csv')
csv_docs

[Document(metadata={'source': 'employees.csv', 'char_count': 26}, page_content='이름: 김철수 | 부서: 개발팀 | 직급: 선임'),
 Document(metadata={'source': 'employees.csv', 'char_count': 27}, page_content='이름: 이영희 | 부서: 기획팀 | 직급: 매니저'),
 Document(metadata={'source': 'employees.csv', 'char_count': 26}, page_content='이름: 박지민 | 부서: 개발팀 | 직급: 주임')]

In [74]:
test_json = {'name' : 'RAG 프로젝트', 'version' : '1.0', 'features' : ['검색', '생성']}
with open('sample_data/test.json', 'w', encoding='utf-8') as f:
    json.dump(test_json, f, ensure_ascii=False)

In [75]:
with open('sample_data/test.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    
content = json.dumps(data, ensure_ascii=False, indent=2)

In [76]:
data

{'name': 'RAG 프로젝트', 'version': '1.0', 'features': ['검색', '생성']}

In [77]:
content

'{\n  "name": "RAG 프로젝트",\n  "version": "1.0",\n  "features": [\n    "검색",\n    "생성"\n  ]\n}'

In [78]:
doc = Document(page_content = content, metadata = {'source' : 'test.json', 'keys' : data.keys()})
doc

Document(metadata={'source': 'test.json', 'keys': dict_keys(['name', 'version', 'features'])}, page_content='{\n  "name": "RAG 프로젝트",\n  "version": "1.0",\n  "features": [\n    "검색",\n    "생성"\n  ]\n}')

In [80]:
def load_json_as_document(file_path):
    path = Path(file_path)
    with open(path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    content = json.dumps(data, ensure_ascii=False, indent=2) 
    keys = list(data.keys())
    return Document(page_content = content, metadata= {'source' : path.name, 'keys' :  keys})